In [1]:
import numpy as np
import os
import sys
from stable_baselines3 import SAC
from stable_baselines3 import DQN
import torch
import pandas as pd

In [2]:
sys.path.insert(0,'boptestGym')
from boptestGymEnv import BoptestGymEnv
from boptestGymEnv import NormalizedObservationWrapper

In [3]:
url = "http://localhost:80"
log_path = os.path.join("Logs")

In [6]:
def create_env():
    env = BoptestGymEnv(url               = url,
                    testcase              = 'bestest_hydronic_heat_pump',
                    actions               = ['oveHeaPumY_u'],
                    observations          = {'reaTZon_y':(273.,324.), #0 / 50
                                             'weaSta_reaWeaTDryBul_y':(257.,305.),
                                             'weaSta_reaWeaHDirNor_y':(0.,862.)
                                            },
                    random_start_time     = False,
                    start_time            = 305*24*3600, #nov 1 like the paper
                    max_episode_length    = 61 * 24*3600, #2 month testing period
                    warmup_period         = 24*3600,
                    predictive_period     = 0,
                    regressive_period     = 4*1800,
                    step_period           = 1800)
    env = NormalizedObservationWrapper(env)
    return env

C:\Users\irmak\anaconda3\Lib\site-packages\gymnasium\spaces\box.py:130: UserWarning: WARN: Box bound precision lowered by casting to float32
  gym.logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [ ]:
for data_regime in [6, 27]:
    for training_episodes in [25, 50]:
        print(f"training episode: {training_episodes}")
        model= SAC.load(f"Saved Models/SAC_switch_1_{data_regime}_{training_episodes}.zip")
        env = create_env()
        done = False
        obs, _ = env.reset()
        rows = []

        i=0
        while i<=2928:
            # Clear the display output at each step
            clear_output(wait=True)
    
            # Compute control signal
            action, _ = model.predict(obs, deterministic=True)
            kpis = requests.get('{0}/kpi/{1}'.format(url, env.testid)).json()['payload']

            if isinstance(action, (tuple, list, np.ndarray)):
                action = int(np.array(action).flatten()[0])
            # Print the current operative temperature and decided action
            print('-------------------------------------------------------------------')
            print("obs: %s"%obs)
            print("act: %s"%action)
            print("%s /2928"%i)
            print('-------------------------------------------------------------------')
            i+=1
            # Implement action
            rows.append({
                "T_zone": obs[0],
                "t_out": obs[1],
                "Psol_Wm2": obs[2],
                "action": action,
                "energy_kWh": kpis["ener_tot"],
                "discomfort": kpis["tdis_tot"]
            })
            obs, reward, terminated, truncated, info = env.step(action)  # send the action to the environment
            done = (terminated or truncated)

        df = pd.DataFrame(rows)
        df.to_csv(f"Testing/SAC_switch_1_{data_regime}_{training_episodes}.csv", index=False)
        env.stop()